In [1]:
"""
Aliquot 16ul qPCR Mix into 100ul qPCR, 96-well plate from 2.0mL tube on OT-2 rack.
This script uses 50ul tips for mastermix dispense and 5oul tips for template addition.
NOT 10UL TIPS!
Add 4ul template from DW source plate. 
This script can be used for bioreactor E. coli dilution studies.

This uses an entire box of 50ul tips.

Author : Harley King
Date   : 2025-10-16

To Begin:
1) Need 50ul tips at position = 1 and 3
2) make sure both methods below are active (not comented out)
3) qPCR MMix (with extra MgCl2) in OT-2 rack at D6, position[1]
4) DW source plate at positiom[0] on rails 19
5) water plate behind qPCR plate on rails 13
"""


'\nAliquot 16ul qPCR Mix into 100ul qPCR, 96-well plate from 2.0mL tube on OT-2 rack.\nThis script uses 50ul tips for mastermix dispense and 5oul tips for template addition.\nNOT 10UL TIPS!\nAdd 4ul template from DW source plate. \nThis script can be used for bioreactor E. coli dilution studies.\n\nThis uses an entire box of 50ul tips.\n\nAuthor : Harley King\nDate   : 2025-10-16\n\nTo Begin:\n1) Need 50ul tips at position = 1 and 3\n2) make sure both methods below are active (not comented out)\n3) qPCR MMix (with extra MgCl2) in OT-2 rack at D6, position[1]\n4) DW source plate at positiom[0] on rails 19\n5) water plate behind qPCR plate on rails 13\n'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.opentrons.tube_racks import (
    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
)
from pylabrobot.resources.eppendorf.tubes import eppendorf_tube_1500uL_Vb

from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL,     # 1000 µL filtered 
    hamilton_96_tiprack_10uL_filter,  # 10 µL filtered
)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

2026-02-23 16:09:23,050 - pylabrobot.io.usb - INFO - Finding USB device...
2026-02-23 16:09:23,058 - pylabrobot.io.usb - INFO - Found USB device.
2026-02-23 16:09:23,061 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-02-23 16:09:26,224 - pylabrobot - INFO - Running backend initialization procedure.


In [4]:
from pylabrobot.resources.plate import Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)
def VWR_96_wellplate_100_Vb(name: str, with_lid: bool = False) -> Plate:
  """
This plate is a VWR PCR plate 96 well low-profile, half-skirted, ABI-FAST type plate.
VWR cat no. 89218-296
It is half-skirted so it must reside in another plate like a Cor_96_wellplate_360ul_Fb
  """
  
  return Plate(
    name=name,
    size_x=127.76,
    size_y=85.48,
    size_z=20.0,
    # lid=lid,
    model=VWR_96_wellplate_100_Vb.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      num_items_x=12,
      num_items_y=8,
      dx=11.5,  # keeping costar measurement
      dy=8.75,  # 7.77 keeping costar measurement
      dz=8.5, # how high is well above base
      item_dx=9.0,
      item_dy=9.0,
      size_x=5.4,  # measured
      size_y=5.4,  # measured
      size_z=16.3, # measured well depth, costar + VWR plate height
      material_z_thickness=0.5,
      bottom_type=WellBottomType.V,
      cross_section_type=CrossSectionType.CIRCLE,
      max_volume=100,
    ),
  )

from typing import Optional

from pylabrobot.resources.height_volume_functions import (
  compute_height_from_volume_rectangle,
  compute_volume_from_height_rectangle,
)
from pylabrobot.resources.plate import Lid, Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

In [5]:
###############################################################################
# 1) carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
# --- tip carrier -------------------------------------------------------------
from pylabrobot.resources import hamilton_96_tiprack_1000uL

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
# tiprack_1000 = HTF("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_1000 = hamilton_96_tiprack_1000uL("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_50   = hamilton_96_tiprack_50uL_filter("tips_01") #  50 µL filter tips (slot-1)
tiprack_50_mastermix   = hamilton_96_tiprack_50uL_filter("tips_03") #  50 µL filter tips (slot-1)
tiprack_10   = hamilton_96_tiprack_10uL_filter("tips_02") #  10 µL filter tips (slot-2)
# mount the racks
tip_car[0] = tiprack_1000         
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10
tip_car[3] = tiprack_50_mastermix



# STANDARDS RACK
dwp_mod_dest   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dest")
car_07 = MFX_CAR_L5_base(
    "car_07",
    modules={
        1: dwp_mod_dest
    }
)
lh.deck.assign_child_resource(car_07, rails=7)

# labware
# tuberack_mix = opentrons_24_tuberack_generic_1point5ml_snapcap_short("mix_rack")
# dest_offset_x = (127.76 - tuberack_mix._size_x) / 2
# dest_offset_y = (85.48  - tuberack_mix._size_y) / 2

# adapter_dest = TubeRackAdapter(
#     name="dest_rack_adapter",
#     size_x=127.76,
#     size_y=85.48,
#     size_z=tuberack_mix._size_z,              # external height of the frame
#     model="tube_rack_adapter",
#     dx=dest_offset_x,
#     dy=dest_offset_y,
#     dz=0,
#     adapter_hole_size_x=tuberack_mix._size_x,
#     adapter_hole_size_y=tuberack_mix._size_y,
#     adapter_hole_size_z=tuberack_mix._size_z
# )
tuberack = opentrons_24_tuberack_generic_1point5ml_snapcap_short("dest_rack")
# adapter_dest.assign_child_resource(tuberack_mix)
dwp_mod_dest.assign_child_resource(tuberack)
# Put mastermix tube into OT2 rack position D6
mastermix_tube = eppendorf_tube_1500uL_Vb("mastermix_tube")

# tuberack["D6"] is a ResourceHolder (a position). Assign the tube into that position.
# tuberack["D6"].assign_child_resource(mastermix_tube)
# holder_D6 = tuberack.get_item("D6")
tuberack.get_item("D6").assign_child_resource(mastermix_tube)

# ----------carrier @ rail 13: water trough-------------------
# 96W, 100ul VWR PCR plate
dwp_mod_PCR = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_PCR")
dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
car_13 = MFX_CAR_L5_base(
    "car_13",
    modules={
        0: dwp_mod_PCR,
        1: dwp_mod_trough,
    }
)
lh.deck.assign_child_resource(car_13, rails=13)
qPCR_plate = VWR_96_wellplate_100_Vb("qPCR_plate")
dwp_mod_PCR.assign_child_resource(qPCR_plate)
trough = AGenBio_1_troughplate_100000uL_Fl("water_trough")
dwp_mod_trough.assign_child_resource(trough)

# --- carrier @ rail-19: VWR 100ul plate containing DNA ----------------------------
dwp_mod_src = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_src")
car_19 = MFX_CAR_L5_base(
    "car_19",
    modules={
        0: dwp_mod_src,
    }
)
lh.deck.assign_child_resource(car_19, rails=19)

plate_src = BioER_96_wellplate_Vb_2200uL("plate_src")


dwp_mod_src.assign_child_resource(plate_src)

/tmp/ipykernel_1352211/1656319453.py:24: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  dwp_mod_dest   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dest")
/tmp/ipykernel_1352211/1656319453.py:25: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_07 = MFX_CAR_L5_base(
/tmp/ipykernel_1352211/1656319453.py:64: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  dwp_mod_PCR = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_PCR")
/tmp/ipykernel_1352211/1656319453.py:65: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
/tmp/ipykernel_1352211/1656319453.py:66: DeprecationWarning: MFX_CAR_L5_base is de

In [6]:
###############################################################################
# 2) high-level parameters  ── adjust here if anything changes later
###############################################################################
CHANNEL_MM   = 4            # single channel we’ll use for the whole run
SRC_MM_TUBE  = [mastermix_tube]  # mastermix (carrier @ rail 7, tube rack “src_rack”)
DEST_PLATE   = qPCR_plate           # 96-well PCR plate (carrier @ rail 13)
START_TIP = "A1"
CHANNELS_8= list(range(8))


###############################################################################
# 3) helper utilities
###############################################################################
def plate_coords(rows, cols):
    """Return well objects for the supplied rows (string of A-H)
       and cols (iterable of ints 1-12)."""
    return [DEST_PLATE[f"{row}{col}"] for row in rows for col in cols]

ROW_LET = "ABCDEFGH"
def tip_sequence(start: str):
    sr, sc = start[0].upper(), int(start[1:])
    for r in ROW_LET[ROW_LET.index(sr):]:
        for c in range(sc if r == sr else 1, 13):
            yield f"{r}{c}"

async def drop_tip(channel):
    await lh.drop_tips(use_channels=[channel])

###############################################################################
# 4) distribute mastermix (18 µL into every well)
###############################################################################
async def dispense_mastermix():
    rows = "ABCDEFGH"
    cols = range(1, 13)          # 1-12
    all_wells = plate_coords(rows, cols)    # todo: change this from 50ul tips in box [03] to 50ul tips in [01] 
    await lh.pick_up_tips(tiprack_50_mastermix["A1"], use_channels=[CHANNEL_MM]) #only 1, 50ul tip needed for dispense_mastermix
    # moisten tip
    await lh.aspirate(
                    SRC_MM_TUBE, 
                    vols=[0],
                    use_channels=[CHANNEL_MM],
                    lld_mode=[STARBackend.LLDMode.GAMMA],
                    # immersion_depth=[1], 
                    # surface_following_distance=[2],
                    # mix_volume=[50],
                    # mix_cycles=[2],
                    # mix_surface_following_distance=[2],
                    settling_time=[2],
                    blow_out=[1]
                )
    # iterate two wells at a time (36 µL = 16+16+4 µL blow-out)
    well_pairs = [all_wells[i:i+2] for i in range(0, len(all_wells), 2)]

    for pair in well_pairs:
        # 1. aspirate 36 µL mastermix
        try: #when liquid height gets too low
            await lh.aspirate(
                SRC_MM_TUBE, vols=[40],
                use_channels=[CHANNEL_MM],
                lld_mode=[STARBackend.LLDMode.GAMMA],
                immersion_depth=[5], 
                surface_following_distance=[2],
                settling_time=[1], 
                transport_air_volume=[0],
                flow_rates=[20]
            )
        except: 
            await lh.aspirate(
                SRC_MM_TUBE, vols=[40], liquid_height=[2],
                use_channels=[CHANNEL_MM],
                settling_time=[1], 
                transport_air_volume=[0]
            )

        # 2. dispense 18 µL into each target well
        for dest in pair:
            await lh.dispense(
                dest, vols=[16],
                liquid_height=[1],
                use_channels=[CHANNEL_MM],
                transport_air_volume=[0],
                settling_time=[1],
                flow_rates=[10]
            )

        # 3. blow-out remaining ~4 µL back to mastermix tube
        try: #when liquid height gets too low
            await lh.dispense(
                SRC_MM_TUBE, vols=[8],   # 0 µL triggers blow-out in PLR
                use_channels=[CHANNEL_MM],
                lld_mode=[STARBackend.LLDMode.GAMMA],
                immersion_depth=[1],
                blow_out=[1]
            )
        except:
            await lh.dispense(
                SRC_MM_TUBE, vols=[8],   # 0 µL triggers blow-out in PLR
                use_channels=[CHANNEL_MM],
                liquid_height=[2],
                blow_out=[1]
            )

    await lh.discard_tips()

async def add_template_with_50ul_tips():
    # begin in column 1 of the plate_src at position[1] on rail 19
    # mix at mid and high z, e.g. 4, 7 mm in a total volume of 20+20=40ul
    for col in range(1,13): 
          # todo: change from 50ul tips to 10ul tips
        # await lh.pick_up_tips(tiprack_10[f"A{col}:H{col}"], use_channels=CHANNELS_8) # all 8 channels should have tips
        await lh.pick_up_tips(tiprack_50[f"A{col}:H{col}"], use_channels=CHANNELS_8) # all 8 channels should have tips
                # aspirate 4ul and add to plate containing mmix

        # todo remove or change the pre-moisten volumes below if using 10ul tips
        # pre-moisten the tips for more accurate dispensing with 50ul filtered tips
        try:
            await lh.aspirate(
                plate_src[f"A{col}:H{col}"],
                vols=[30]*8,
                use_channels=CHANNELS_8,
                lld_mode=[STARBackend.LLDMode.GAMMA]*8,
                transport_air_volume=[0]*8,    
                immersion_depth=[1]*8
            )
            await lh.dispense(
                plate_src[f"A{col}:H{col}"],
                vols=[30]*8,
                use_channels=CHANNELS_8,
                lld_mode=[STARBackend.LLDMode.GAMMA]*8,
                transport_air_volume=[0]*8,
                blow_out=[1]*8, 
                settling_time=[1]*8
            )
        except:
            print ("pre-wettening step failed due to: ")

        # the pipetting step:
        await lh.aspirate(
            plate_src[f"A{col}:H{col}"],
            vols=[10]*8,
            use_channels=CHANNELS_8,
            # liquid_height=[4]*8,
            lld_mode=[STARBackend.LLDMode.GAMMA]*8,
            transport_air_volume=[0]*8,    
            immersion_depth=[1]*8
        )
        # todo remove the prime step below
        # prime the 50ul tips
        await lh.dispense(
            plate_src[f"A{col}:H{col}"], #return a few ul back to source
            vols=[4]*8,
            use_channels=CHANNELS_8,
            lld_mode=[STARBackend.LLDMode.GAMMA]*8,
            immerse_depth=[1]*8,
            transport_air_volume=[0]*8,
        )
        # actual dispense to qPCR plate with moistened, primed 50ul tips
        await lh.dispense(
            qPCR_plate[f"A{col}:H{col}"],
            vols=[4]*8,
            use_channels=CHANNELS_8,
            liquid_height=[1]*8,
            transport_air_volume=[0]*8,
            flow_rates=[4]*8,
            # blow_out=[1]*8, 
            settling_time=[1]*8
        )
        await lh.discard_tips()


    


In [7]:
await dispense_mastermix()
await add_template_with_50ul_tips()

/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot/pylabrobot/liquid_handling/liquid_handler.py:345: UserWarning: Extra arguments to backend.dispense: {'immerse_depth'}
  warnings.warn(f"Extra arguments to backend.{method.__name__}: {extra}")


In [8]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.dispense(SRC_MM_TUBE, vols=[2], liquid_height=[2], use_channels=[CHANNEL_MM])
# await lh.drop_tips(tiprack_10["A1:H1"], use_channels=CHANNELS_8)
# await lh.dispense(
#             qPCR_plate["A11:H11"],
#             vols=[4]*8,
#             use_channels=CHANNELS_8,
#             liquid_height=[1]*8,
#             transport_air_volume=[0]*8,
#             flow_rates=[4]*8,
#             blow_out=[1]*8, 
#             settling_time=[1]*8
#             )
# await lh.discard_tips()
# await lh.stop()
# print(type(tuberack))
# print(type(tuberack.ordered_items))
# print(list(tuberack.ordered_items.keys())[:8])  # if dict
# print(tuberack.ordered_items.get("D6"))

# print([a for a in dir(tuberack) if "ordered" in a.lower() or "items" in a.lower()][:50])
# print([a for a in dir(tuberack) if a.startswith("_") and "ordered" in a.lower()][:50])
# print([a for a in dir(tuberack) if a.startswith("_") and "items" in a.lower()][:50])




